# Phase A (Notebook 1): Animated Character & Visual Pipeline (`PhaseA1_Visual_Pipeline.ipynb`)

**Environment Stack**: `Animated_AD/conda.yaml` Visual Stack (`PySceneDetect`, `OWLv2`, `SAM 2.1` + `xformers`, `DINOv2` on `PyTorch 2.5.1 + cu124`).

### Workflow Overview
1. **Cell 1 — Hardware Check, Google Drive Mount & Persistent Model Cache Setup**:
   - Mounts Google Drive (`/content/drive/MyDrive/MovieRAG`) for **movie inputs, persistent model weights (`models_cache/`), and output checkpoints (`workdir/{movie_id}/`)**.
   - Pins `HF_HOME`, `TORCH_HOME`, and `SAM2_CHECKPOINT` directly to Google Drive so `SAM 2.1`, `OWLv2`, and `DINOv2` are downloaded **only once** and reused immediately on every future Colab session.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag`)**:
   - Clones your GitHub repository into `/content/MovieRag` (or pulls the latest commits if already cloned).
3. **Cell 3 — Install Visual Stack Dependencies**:
   - Installs `torch==2.5.1+cu124`, `xformers`, `sam2`, `scenedetect[opencv]`, `decord`, and `transformers`.
4. **Cell 4 — Persistent Drive Model Download & Verification**:
   - Verifies or downloads `SAM 2.1` (`sam2.1_hiera_large.pt`), `OWLv2` (`google/owlv2-large-patch14-ensemble`), and `DINOv2` (`dinov2_vitg14_reg` / `dinov2_vitl14_reg`) into `/content/drive/MyDrive/MovieRAG/models_cache/`.
5. **Cell 5 — Stage A1 (`stage1_scenes.py`)**:
   - Shot boundary detection, subdivision of shots $>20\text{s}$ into $15\text{s}$ sub-shots, and grouping into $15\text{--}30\text{s}$ Scene Windows (`shots_and_scenes.json`).
6. **Cell 6 — Stage A2 (`stage2_charbank.py`)**:
   - 75-epoch `DINOv2` contrastive fine-tuning on `char_bank/{movie_id}/` and exemplar `.npz` feature extraction.
7. **Cell 7 — Stage A3 (`stage3_visual_recog.py`)**:
   - Per-shot `TGRP` tracking (resumable via `tgrp_tracks.jsonl`), `DINOv2` track classification (`visual_tracks_per_frame.json`), and **Adaptive Per-Shot Sharp Keyframe Extraction** with pre-drawn character bounding boxes (`keyframes/*.jpg`).
8. **Cell 8 — Handoff Verification & Inline Visualization**.

In [ ]:
# ==============================================================================
# CELL 1: Hardware Check, Google Drive Mount & Persistent Model Cache Setup
# ==============================================================================
import os
import subprocess

# 1. Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

# 2. Configure Movie & Google Drive Paths
MOVIE_ID = "sintel"                       # @param {type:"string"}
MOVIE_TITLE = "Sintel"                    # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"  # @param {type:"string"}

VIDEO_PATH = f"{DRIVE_ROOT}/videos/{MOVIE_ID}.mp4"
CHAR_BANK_DIR = f"{DRIVE_ROOT}/char_bank/{MOVIE_ID}"
WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"

# 3. Configure Persistent Model Cache on Google Drive
#    All 3 foundation models (SAM 2.1, OWLv2, DINOv2) are stored on Drive so
#    future Colab sessions never have to re-download them.
MODELS_CACHE_DIR = f"{DRIVE_ROOT}/models_cache"
HF_CACHE_DIR = f"{MODELS_CACHE_DIR}/huggingface"
TORCH_HUB_CACHE_DIR = f"{MODELS_CACHE_DIR}/torch_hub"
SAM2_CACHE_DIR = f"{MODELS_CACHE_DIR}/sam2"
SAM2_CHECKPOINT = f"{SAM2_CACHE_DIR}/sam2.1_hiera_large.pt"

for d in (WORKDIR, HF_CACHE_DIR, TORCH_HUB_CACHE_DIR, SAM2_CACHE_DIR):
    os.makedirs(d, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = f"{HF_CACHE_DIR}/hub"
os.environ["TORCH_HOME"] = TORCH_HUB_CACHE_DIR

# 4. Detect GPU Tier (L4 24GB -> 'giant', T4 16GB -> 'large')
gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True)
if gpu_info.returncode == 0 and gpu_info.stdout.strip():
    gpu_line = gpu_info.stdout.strip().splitlines()[0]
    print(f"Detected GPU: {gpu_line}")
    if "L4" in gpu_line or "A100" in gpu_line or "24" in gpu_line or "40" in gpu_line or "80" in gpu_line:
        MODEL_SIZE = "giant"
    else:
        MODEL_SIZE = "large"
else:
    print("WARNING: No NVIDIA GPU detected! Make sure Colab runtime is set to GPU (L4 or T4).")
    MODEL_SIZE = "large"

print(f"Selected DINOv2 MODEL_SIZE = '{MODEL_SIZE}'")
print(f"Video Path     : {VIDEO_PATH}")
print(f"Char Bank Dir  : {CHAR_BANK_DIR}")
print(f"Output Workdir : {WORKDIR}")
print(f"Drive Models   : {MODELS_CACHE_DIR}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag)
# ==============================================================================
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/YOUR_USERNAME/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                             # @param {type:"string"}
REPO_SUBDIR = ""  # Set to "MovieRag" if your repo root is ctu-flix monorepo, or "" if MovieRag is the repo root
CLONE_DIR = "/content/MovieRag_repo"

# Optional: Load GITHUB_TOKEN from Colab Secrets if the repository is private
github_token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH}) into {CLONE_DIR}...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Repository already exists at {CLONE_DIR}; pulling latest changes from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

# Auto-detect whether MovieRag is at CLONE_DIR or CLONE_DIR/MovieRag
if REPO_SUBDIR and os.path.isdir(os.path.join(CLONE_DIR, REPO_SUBDIR)):
    PROJECT_ROOT = os.path.join(CLONE_DIR, REPO_SUBDIR)
elif os.path.isdir(os.path.join(CLONE_DIR, "MovieRag", "pipeline")):
    PROJECT_ROOT = os.path.join(CLONE_DIR, "MovieRag")
else:
    PROJECT_ROOT = CLONE_DIR

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.environ["PYTHONPATH"] = f"{PROJECT_ROOT}:{PROJECT_ROOT}/Animated_AD:" + os.environ.get("PYTHONPATH", "")

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Latest Commit      : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Animated_AD Visual Environment Stack
# ==============================================================================
# Matches Animated_AD/conda.yaml:
# - PyTorch 2.5.1 + torchvision 0.20.1 (CUDA 12.4)
# - xformers, SAM 2.1, PySceneDetect, Decord, OWLv2 (transformers)

!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install -q xformers --index-url https://download.pytorch.org/whl/cu124
!pip install -q omegaconf torchmetrics fvcore iopath decord "scenedetect[opencv]" transformers accelerate huggingface_hub networkx scipy scikit-learn matplotlib pillow tqdm

import os
if not os.path.isdir("/content/sam2"):
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2
    !pip install -q -e /content/sam2 --no-build-isolation

print("Visual environment packages installed successfully.")

In [ ]:
# ==============================================================================
# CELL 4: Persistent Drive Model Download & Cache Verification
# ==============================================================================
# Downloads SAM 2.1, OWLv2, and DINOv2 directly into /content/drive/MyDrive/MovieRAG/models_cache/
# on the first run. On all subsequent Colab sessions, it detects the cached weights on Drive
# and skips downloading.

import gc
import os
import urllib.request
import torch
from transformers import Owlv2ForObjectDetection, Owlv2Processor

torch.hub.set_dir(TORCH_HUB_CACHE_DIR)

# 1. SAM 2.1 Large Checkpoint (~898 MB)
SAM2_URL = "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt"
if os.path.isfile(SAM2_CHECKPOINT) and os.path.getsize(SAM2_CHECKPOINT) > 800 * 1024 * 1024:
    size_mb = os.path.getsize(SAM2_CHECKPOINT) / (1024 * 1024)
    print(f"[1/3] SAM 2.1 already cached on Google Drive: {SAM2_CHECKPOINT} ({size_mb:.1f} MB)")
else:
    print(f"[1/3] Downloading SAM 2.1 checkpoint to Google Drive: {SAM2_CHECKPOINT} ...")
    urllib.request.urlretrieve(SAM2_URL, SAM2_CHECKPOINT)
    size_mb = os.path.getsize(SAM2_CHECKPOINT) / (1024 * 1024)
    print(f"      -> Saved SAM 2.1 checkpoint ({size_mb:.1f} MB)")

# 2. OWLv2 (google/owlv2-large-patch14-ensemble -> cached under HF_HOME on Drive)
print(f"[2/3] Verifying / caching OWLv2 in Drive HF_HOME ({HF_CACHE_DIR})...")
_owl_proc = Owlv2Processor.from_pretrained("google/owlv2-large-patch14-ensemble")
_owl_model = Owlv2ForObjectDetection.from_pretrained("google/owlv2-large-patch14-ensemble")
del _owl_proc, _owl_model
print("      -> OWLv2 ready on Google Drive.")

# 3. DINOv2 (facebookresearch/dinov2 -> cached under TORCH_HOME on Drive)
dinov2_arch = {"small": "vits14", "base": "vitb14", "large": "vitl14", "giant": "vitg14"}[MODEL_SIZE]
dinov2_model_name = f"dinov2_{dinov2_arch}_reg"
print(f"[3/3] Verifying / caching DINOv2 ({dinov2_model_name}) in Drive TORCH_HOME ({TORCH_HUB_CACHE_DIR})...")
_dino_model = torch.hub.load("facebookresearch/dinov2", dinov2_model_name, pretrained=True, verbose=False)
del _dino_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print(f"      -> {dinov2_model_name} ready on Google Drive.")
print("\nAll 3 foundation models are cached on Google Drive and verified!")

In [ ]:
# ==============================================================================
# CELL 5: Stage A1 — Shot Boundary Detection & 15-30s Scene Windowing
# ==============================================================================
import json
from pipeline.stage1_scenes import run_stage1_scenes

stage1_result = run_stage1_scenes(
    video_path=VIDEO_PATH,
    movie_id=MOVIE_ID,
    output_dir=WORKDIR,
    movie_title=MOVIE_TITLE,
    min_scene_sec=12.0,
    max_scene_sec=30.0,
    max_shot_sec=20.0,
    sub_shot_sec=15.0,
)

print("\n--- Stage A1 Summary ---")
print(f"Raw Detected Shots : {stage1_result['num_raw_shots']}")
print(f"Subdivided (>20s)  : {stage1_result['num_subdivided_shots']}")
print(f"Canonical Shots    : {stage1_result['num_shots']}")
print(f"Grouped Scenes     : {stage1_result['num_scenes']}")
if stage1_result["scenes"]:
    print("Sample Scene 0     :", json.dumps(stage1_result["scenes"][0], indent=2))

In [ ]:
# ==============================================================================
# CELL 6: Stage A2 — Visual Character Bank & 75-Epoch DINOv2 Fine-Tuning
# ==============================================================================
from pipeline.stage2_charbank import run_stage2_charbank

OVERWRITE_DINOV2_FINETUNE = False  # @param {type:"boolean"}

stage2_result = run_stage2_charbank(
    movie_id=MOVIE_ID,
    char_bank_dir=CHAR_BANK_DIR,
    output_dir=WORKDIR,
    model_size=MODEL_SIZE,
    epochs=75,
    learning_rate=6e-4,
    temperature=0.07,
    num_per_character=8,
    use_box=True,
    use_mask=False,
    sam2_checkpoint=SAM2_CHECKPOINT,
    overwrite=OVERWRITE_DINOV2_FINETUNE,
)

CKPT_WEIGHTS_PATH = stage2_result["ckpt_weights_path"]
CHAR_FEATURES_DIR = stage2_result["char_features_dir"]
print("\n--- Stage A2 Summary ---")
print(f"Fine-Tuned Weights : {CKPT_WEIGHTS_PATH}")
print(f"Character Bank Dir : {CHAR_FEATURES_DIR}")
print(f"Characters Indexed : {stage2_result['characters']}")

In [ ]:
# ==============================================================================
# CELL 7: Stage A3 — Per-Shot TGRP Tracking, DINOv2 ID & Adaptive Annotated Keyframes
# ==============================================================================
import os
from pipeline.stage3_visual_recog import run_stage3_visual_recog

SHOTS_FILE = os.path.join(WORKDIR, "shots_and_scenes.json")
RESTART_TGRP_FROM_SCRATCH = False  # @param {type:"boolean"}

stage3_result = run_stage3_visual_recog(
    video_path=VIDEO_PATH,
    shots_file=SHOTS_FILE,
    ckpt_weights=CKPT_WEIGHTS_PATH,
    char_features_dir=CHAR_FEATURES_DIR,
    sam2_checkpoint=SAM2_CHECKPOINT,
    output_dir=WORKDIR,
    model_size=MODEL_SIZE,
    mask_classify=False,
    visual_prompt_thresh=0.45,
    voice_harvest_thresh=0.60,
    restart=RESTART_TGRP_FROM_SCRATCH,
)

print("\n--- Stage A3 Summary ---")
print(f"Total Shots Tracked        : {stage3_result['num_shots']}")
print(f"Total Scenes Updated       : {stage3_result['num_scenes']}")
print(f"Total Annotated Keyframes  : {stage3_result['total_keyframes']}")
print(f"Track Classification Stats : {stage3_result['track_stats']}")

In [ ]:
# ==============================================================================
# CELL 8: Handoff Artifact Verification & Sample Annotated Keyframe Inspection
# ==============================================================================
import json
import os
import matplotlib.pyplot as plt
from PIL import Image

required_files = [
    os.path.join(WORKDIR, "shots_and_scenes.json"),
    os.path.join(WORKDIR, "visual_tracks_per_frame.json"),
    os.path.join(WORKDIR, "classified_tracks.json"),
    os.path.join(WORKDIR, "character_glossary.json"),
]

print("=== Notebook 1 -> Notebook 2 Handoff Verification ===")
for path in required_files:
    exists = os.path.isfile(path)
    size_kb = os.path.getsize(path) / 1024.0 if exists else 0.0
    print(f"[{'OK' if exists else 'MISSING'}] {path} ({size_kb:.1f} KB)")

with open(os.path.join(WORKDIR, "shots_and_scenes.json"), "r", encoding="utf-8") as f:
    scenes_data = json.load(f)

# Collect up to 4 annotated keyframes that have visible character bounding boxes
annotated_samples = []
for scene in scenes_data.get("scenes", []):
    for kf in scene.get("keyframes", []):
        if kf.get("characters_visible"):
            annotated_samples.append((scene["scene_id"], kf))
            if len(annotated_samples) >= 4:
                break
    if len(annotated_samples) >= 4:
        break

if annotated_samples:
    fig, axes = plt.subplots(1, len(annotated_samples), figsize=(5 * len(annotated_samples), 4))
    if len(annotated_samples) == 1:
        axes = [axes]
    for ax, (scene_id, kf) in zip(axes, annotated_samples):
        img_path = os.path.join(WORKDIR, kf["path"])
        if os.path.isfile(img_path):
            ax.imshow(Image.open(img_path))
        legend_str = ", ".join(
            f"{c['name']} ({c['color']}: {c['score']:.2f})" for c in kf["characters_visible"]
        )
        ax.set_title(f"{scene_id} | t={kf['timestamp']:.2f}s\n{legend_str}", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No keyframes with character detections above threshold found to display.")